In [1]:
import os
import re
import math
import pandas as pd

def highlight_close_distances(val):
    """
    Returns a CSS string to style cells green if the distance is less than 4 meters.
    """
    if isinstance(val, (int, float)) and val < 4:
        return 'background-color: #c8e6c9; color: #1b5e20; font-weight: bold;'  # Soft green background with dark green text
    return ''

# Wezep

In [2]:
# Borehole data Wezep
borehole_data = {
    'Borehole_ID': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    'X': [196014, 195980, 195998, 195988, 196005, 195986, 195999, 195979, 196001, 195992, 195994, 195985],
    'Y': [497413, 497435, 497401, 497411, 497415, 497440, 497410, 497424, 497422, 497434, 497418, 497429]
}

df_boreholes = pd.DataFrame(borehole_data)

# Change this to the path where your .gef files are stored
gef_folder_path = r"C:\AA_Thesis\CPT_Measurements\Wezep"  

cpt_coordinates = {}

# Process each file in the folder
for filename in os.listdir(gef_folder_path):
    if filename.endswith('.gef'):
        # Extract CPT ID using regex (matches everything between the last '_' and '.gef')
        match_id = re.search(r'_([A-Za-z0-9]+)\.gef$', filename)
        if not match_id:
            continue
        cpt_id = match_id.group(1)
        
        filepath = os.path.join(gef_folder_path, filename)
        
        # Read file to find the #XYID line
        with open(filepath, 'r', encoding='latin-1') as file:
            for line in file:
                if line.startswith('#XYID='):
                    # Clean line and split by commas
                    parts = line.replace('#XYID=', '').strip().split(',')
                    if len(parts) >= 3:
                        try:
                            # 2nd value is X, 3rd value is Y
                            cpt_x = float(parts[1].strip())
                            cpt_y = float(parts[2].strip())
                            cpt_coordinates[cpt_id] = (cpt_x, cpt_y)
                        except ValueError:
                            print(f"Could not parse coordinates in file: {filename}")
                    break

# Initialize the final distance matrix dataframe
# Sorting columns naturally (S1, S2... S15)
sorted_cpt_ids = sorted(cpt_coordinates.keys(), key=lambda x: [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', x)])
df_distances = pd.DataFrame(index=df_boreholes['Borehole_ID'])

# Calculate absolute Euclidean distances
for cpt_id in sorted_cpt_ids:
    cpt_x, cpt_y = cpt_coordinates[cpt_id]
    
    distances = []
    for _, bh in df_boreholes.iterrows():
        # Euclidean distance formula
        dist = math.sqrt((bh['X'] - cpt_x)**2 + (bh['Y'] - cpt_y)**2)
        distances.append(round(dist, 2)) # Rounding to 2 decimals for readability
        
    df_distances[cpt_id] = distances

# print(cpt_coordinates)

# Reset index to make 'Borehole_ID' the first explicit column
df_distances = df_distances.reset_index()

# Apply the styling only to the CPT columns (excluding 'Borehole_ID')
# Note: Use .map() for newer pandas versions (>= 2.0). 
# If you are on an older version of pandas and get an error, change .map to .applymap
styled_distances = df_distances.style.map(highlight_close_distances, subset=sorted_cpt_ids).format("{:.2f}", subset=sorted_cpt_ids)

# Display the styled dataframe directly in your Jupyter Notebook
print("If multiple points in a row it means it is within the pre-determined radius")
styled_distances

If multiple points in a row it means it is within the pre-determined radius


,Borehole_ID,S1,S2,S3,S3A,S4,S5,S6,S7,S8,S9,S10,S11,S12,S13,S14,S15
0,1,45.28,41.49,32.57,33.39,28.69,22.23,14.72,18.48,3.56,41.55,38.54,29.92,18.34,14.42,18.28,8.16
1,2,5.62,16.51,8.72,8.73,16.52,22.62,26.33,35.54,38.75,2.45,12.34,14.63,22.75,26.78,28.84,32.51
2,3,41.33,46.99,33.69,35.07,22.12,31.29,17.25,3.22,22.38,38.31,42.64,34.95,18.90,23.48,10.13,20.26
3,4,27.53,36.55,22.35,23.83,8.85,25.46,14.12,11.88,26.67,24.78,31.84,25.66,12.63,20.58,7.84,21.45
4,5,36.53,35.39,24.63,25.64,19.47,17.17,5.72,13.00,9.21,32.86,31.80,23.24,9.37,8.66,10.04,4.72
5,6,12.68,8.89,7.14,5.86,20.36,18.27,26.09,37.67,36.58,10.25,4.53,9.58,22.93,24.52,30.72,30.74
6,7,35.03,38.31,25.57,26.86,16.11,22.24,8.20,5.92,16.45,31.66,34.13,26.16,10.07,14.61,3.29,12.48
7,8,11.86,26.27,13.37,14.55,8.32,24.74,22.00,27.26,35.86,9.71,21.56,19.56,18.46,25.25,21.34,29.56
8,9,29.59,27.39,16.83,17.75,15.10,10.08,3.97,18.08,14.17,25.83,23.74,15.20,4.54,3.64,12.24,7.84
9,10,17.64,13.49,3.50,3.37,15.53,10.61,17.88,30.34,28.14,14.05,9.13,3.16,14.99,16.05,23.42,22.26


# Zwolle

In [3]:
# Borehole data Zwolle
borehole_data = {
    'Borehole_ID': [
        25, 26, 52, 55, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
        14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 27, 28, 29, 30, 31, 32,
        33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
        51, 53, 54, 56, 57, 58, 59, 60
    ],
    'X': [
        202705, 202722, 202739, 202746, 202706, 202678, 202676, 202697, 202668, 202670, 202664, 202672, 202688, 202689, 202691, 202690, 202678,
        202677, 202687, 202667, 202673, 202664, 202680, 202677, 202695, 202683, 202692, 202699, 202695, 202704, 202712, 202722, 202716, 202708,
        202723, 202744, 202738, 202731, 202727, 202715, 202737, 202702, 202711, 202731, 202712, 202718, 202726, 202719, 202733, 202739, 202747, 202743,
        202740, 202698, 202704, 202729, 202731, 202738, 202683, 202705
    ],
    'Y': [
        503448, 503439, 503438, 503432, 503455, 503452, 503461, 503444, 503457, 503441, 503448, 503447, 503449, 503457, 503443, 503436, 503443,
        503435, 503431, 503434, 503428, 503428, 503429, 503420, 503424, 503436, 503417, 503435, 503452, 503440, 503438, 503431, 503449, 503432,
        503449, 503443, 503449, 503448, 503442, 503430, 503432, 503428, 503423, 503419, 503409, 503419, 503412, 503405, 503411, 503419, 503421, 503412,
        503426, 503415, 503420, 503433, 503425, 503444, 503421, 503409
    ]
}

df_boreholes = pd.DataFrame(borehole_data)

# Change this to the path where your .gef files are stored
gef_folder_path = r"C:\AA_Thesis\CPT_Measurements\Zwolle"  

cpt_coordinates = {}

# Process each file in the folder
for filename in os.listdir(gef_folder_path):
    if filename.endswith('.gef'):
        # Extract CPT ID using regex (matches everything between the last '_' and '.gef')
        match_id = re.search(r'_([A-Za-z0-9]+)\.gef$', filename)
        if not match_id:
            continue
        cpt_id = match_id.group(1)
        
        filepath = os.path.join(gef_folder_path, filename)
        
        # Read file to find the #XYID line
        with open(filepath, 'r', encoding='latin-1') as file:
            for line in file:
                if line.startswith('#XYID='):
                    # Clean line and split by commas
                    parts = line.replace('#XYID=', '').strip().split(',')
                    if len(parts) >= 3:
                        try:
                            # 2nd value is X, 3rd value is Y
                            cpt_x = float(parts[1].strip())
                            cpt_y = float(parts[2].strip())
                            cpt_coordinates[cpt_id] = (cpt_x, cpt_y)
                        except ValueError:
                            print(f"Could not parse coordinates in file: {filename}")
                    break

# Initialize the final distance matrix dataframe
# Sorting columns naturally (S1, S2... S15)
sorted_cpt_ids = sorted(cpt_coordinates.keys(), key=lambda x: [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', x)])
df_distances = pd.DataFrame(index=df_boreholes['Borehole_ID'])

# Calculate absolute Euclidean distances
for cpt_id in sorted_cpt_ids:
    cpt_x, cpt_y = cpt_coordinates[cpt_id]
    
    distances = []
    for _, bh in df_boreholes.iterrows():
        # Euclidean distance formula
        dist = math.sqrt((bh['X'] - cpt_x)**2 + (bh['Y'] - cpt_y)**2)
        distances.append(round(dist, 2)) # Rounding to 2 decimals for readability
        
    df_distances[cpt_id] = distances

print(cpt_coordinates)

# Reset index to make 'Borehole_ID' the first explicit column
df_distances = df_distances.reset_index()

# Apply the styling only to the CPT columns (excluding 'Borehole_ID')
# Note: Use .map() for newer pandas versions (>= 2.0). 
# If you are on an older version of pandas and get an error, change .map to .applymap
styled_distances = df_distances.style.map(highlight_close_distances, subset=sorted_cpt_ids).format("{:.2f}", subset=sorted_cpt_ids)

# Display the styled dataframe directly in your Jupyter Notebook
# print("If multiple points in a row it means it is within the pre-determined radius")
# styled_distances

{}
